<div align="center" style="background: linear-gradient(135deg, #1f2937, #111827); padding: 30px; border-radius: 12px; color: white; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <h1 style="margin: 0; font-size: 28px; font-weight: 800; letter-spacing: -0.5px;">🏰 Dar Yacout: BERT Sentiment Analysis</h1>
  <p style="margin: 10px 0 20px 0; font-size: 15px; color: #9ca3af;">End-to-End Multilingual Sentiment Classification Pipeline for Dar Yacout Reviews</p>
  <div>
    <img src="https://img.shields.io/badge/Python-3.12-3776AB?style=for-the-badge&logo=python&logoColor=white" />
    <img src="https://img.shields.io/badge/PyTorch-EE4C2C?style=for-the-badge&logo=pytorch&logoColor=white" />
    <img src="https://img.shields.io/badge/Hugging%20Face-FFD21E?style=for-the-badge&logo=huggingface&logoColor=black" />
    <img src="https://img.shields.io/badge/BERT-Multilingual-blue?style=for-the-badge" />
  </div>
  <img src="image.jpg" width="800" height="500">
</div>

<div align="center">
  
</div>

# 1- Setup

In [ ]:
%pip install torch torchvision torchaudio transformers requests beautifulsoup4 pandas numpy

In [38]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm.notebook import tqdm

import re, time, random
import pandas as pd
import numpy as np

# 2- Model Setup

In [3]:
model_id = 'nlptown/bert-base-multilingual-uncased-sentiment'
tokenizer = AutoTokenizer.from_pretrained(model_id, force_download=True)
model = AutoModelForSequenceClassification.from_pretrained(model_id, force_download=True)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
sentence = 'i hate this, this is absolutely the worst'

encoded_tokens = tokenizer.encode(sentence, return_tensors='pt')
decoded_tokens = tokenizer.decode(encoded_tokens)

print(f'Encoded: {encoded_tokens[0].tolist()}\nDecoded: {decoded_tokens[0]}')

Encoded: [101, 151, 39487, 10372, 117, 10372, 10127, 35925, 10563, 10103, 43060, 102]
Decoded: [CLS] i hate this, this is absolutely the worst [SEP]


In [5]:
result = model(encoded_tokens)
result.logits

tensor([[ 5.1298,  1.5584, -1.0117, -2.9989, -1.8584]],
       grad_fn=<AddmmBackward0>)

In [6]:
int(torch.argmax(result.logits))+1

1

In [7]:
sentence = 'love this'
encoded_tokens = tokenizer.encode(sentence, return_tensors='pt')
result = model(encoded_tokens)


rate = int(torch.argmax(result.logits))+1

print(f'Opinion: {sentence}\nRate: {rate}/5')

Opinion: love this
Rate: 5/5


# 3- Data Collecting

In [ ]:
from curl_cffi import requests as cr
from bs4 import BeautifulSoup

session = cr.Session(impersonate="chrome")
session.get('https://restaurantguru.com/Dar-Yacout-Marrakesh/')
headers={'Referer': 'https://restaurantguru.com/Dar-Yacout-Marrakesh/'}

all_reviews = []

for i in range (1, 70):
    response = session.get(f'https://restaurantguru.com/Dar-Yacout-Marrakesh/reviews?bylang={i}', headers= headers)
    print(f'PAGE: {i} --> response: {response.status_code}')
    if response.status_code != 200:
            print('Blocked, rerun later.')
            break

    soup = BeautifulSoup(response.text, 'html.parser')
    results = soup.find_all('span', {"class": re.compile('.*text_full.*')})
    
    
    page_reviews = [result.text for result in results]
    all_reviews.extend(page_reviews)
    time.sleep(random.uniform(3, 8))
    
    
print(len(all_reviews))
df = pd.DataFrame(np.array(all_reviews), columns=['review'])

In [32]:
df.head(10)

,review
0,Good. Order type: Dine in Vegetarian options: ...
1,Food: 5
2,Food: 4
3,Dining at Dar Yacout was one of the highlights...
4,Spectacular experience: a great sampling of Mo...
5,Price per person: MAD 500+ Parking space: Diff...
6,Wonderful experience in an amazing location.
7,Price per person: MAD 500+
8,"The location is beautiful, but everything abou..."
9,Dar Yacout is undeniably stunning. The archite...


# 4- Pipeline

In [34]:
def scoring_pipeline(review):
    encoded_tokens = tokenizer.encode(review, return_tensors='pt')
    result = model(encoded_tokens)
    rate = int(torch.argmax(result.logits))+1

    return rate

In [36]:


tqdm.pandas(desc="Scoring reviews")

df["sentiment"] = df["review"].progress_apply(lambda x: scoring_pipeline(x[:512]))

Scoring reviews:   0%|          | 0/204 [00:00<?, ?it/s]

In [39]:
df

,review,sentiment
0,Good. Order type: Dine in Vegetarian options: ...,4
1,Food: 5,4
2,Food: 4,4
3,Dining at Dar Yacout was one of the highlights...,5
4,Spectacular experience: a great sampling of Mo...,5
...,...,...
199,It's one of the most amazing restaurants I've ...,5
200,"Magnificent place. Good, very hearty cuisine. ...",5
201,Very good restaurant,5
202,Still as good Vegetarian options: On demand Di...,4


In [41]:
df.to_csv('dar_yacout_reviews_classification.csv', index=False)

# END